In [10]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

In [11]:
nltk.download("vader_lexicon")
nltk.download("stopwords")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
df = pd.read_csv(r"/content/blogs.csv")
df.head()

,Data,Labels
0,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...,alt.atheism
1,Newsgroups: alt.atheism\nPath: cantaloupe.srv....,alt.atheism
2,Path: cantaloupe.srv.cs.cmu.edu!das-news.harva...,alt.atheism
3,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...,alt.atheism
4,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:53...,alt.atheism


In [13]:
df.shape

(2000, 2)

In [14]:
df.columns

Index(['Data', 'Labels'], dtype='object')

In [15]:
df.isnull().sum()

,0
Data,0
Labels,0


In [16]:
df['Labels'].value_counts()

,count
Labels,
alt.atheism,100
comp.graphics,100
comp.os.ms-windows.misc,100
comp.sys.ibm.pc.hardware,100
comp.sys.mac.hardware,100
comp.windows.x,100
misc.forsale,100
rec.autos,100
rec.motorcycles,100


In [17]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.split()
    text = [word for word in text if word not in stop_words]
    return ' '.join(text)

In [18]:
df['clean_text'] = df['Data'].apply(clean_text)
df[['Data','clean_text']].head()

,Data,clean_text
0,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...,path cantaloupesrvcscmuedumagnesiumclubcccmued...
1,Newsgroups: alt.atheism\nPath: cantaloupe.srv....,newsgroups altatheism path cantaloupesrvcscmue...
2,Path: cantaloupe.srv.cs.cmu.edu!das-news.harva...,path cantaloupesrvcscmuedudasnewsharvardedunoc...
3,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...,path cantaloupesrvcscmuedumagnesiumclubcccmued...
4,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:53...,xref cantaloupesrvcscmuedu altatheism talkreli...


In [19]:
tfidf = TfidfVectorizer(max_features=5000)
x = tfidf.fit_transform(df['clean_text'])
y = df['Labels']
x.shape

(2000, 5000)

In [20]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_train.shape, x_test.shape

((1600, 5000), (400, 5000))

In [21]:
model = MultinomialNB()
model.fit(x_train, y_train)

MultinomialNB()

In [22]:
pred = model.predict(x_test)
print("Accuracy Score :", accuracy_score(y_test,pred))
print("\nClassification Report : \n",classification_report(y_test,pred))

Accuracy Score : 0.85

Classification Report : 
                           precision    recall  f1-score   support

             alt.atheism       0.84      0.80      0.82        20
           comp.graphics       0.89      0.85      0.87        20
 comp.os.ms-windows.misc       0.84      0.80      0.82        20
comp.sys.ibm.pc.hardware       0.58      0.75      0.65        20
   comp.sys.mac.hardware       0.83      0.75      0.79        20
          comp.windows.x       0.84      0.80      0.82        20
            misc.forsale       0.86      0.90      0.88        20
               rec.autos       0.86      0.95      0.90        20
         rec.motorcycles       0.94      0.85      0.89        20
      rec.sport.baseball       1.00      0.95      0.97        20
        rec.sport.hockey       1.00      1.00      1.00        20
               sci.crypt       0.87      1.00      0.93        20
         sci.electronics       0.84      0.80      0.82        20
                 sci.med  

In [29]:
"""
GREAT RESULTS THE MODEL PREDICTS THE LABELS WITH THE ACCURACY OF 85%. THE SCORES OF ALL 20 NEWSPAPERS ARE
GREAT WITH SOME HAVING 100 TO 97 PERCENTAGE OF F1 SCORES, SOME NEWSPAPER SHOW LOW SCORES INDICATING OVERLAP IN VOCABULARY
"""

'\nGREAT RESULTS THE MODEL PREDICTS THE LABELS WITH THE ACCURACY OF 85%. THE SCORES OF ALL 20 NEWSPAPERS ARE \nGREAT WITH SOME HAVING 100 TO 97 PERCENTAGE OF F1 SCORES, SOME NEWSPAPER SHOW LOW SCORES INDICATING OVERLAP IN VOCABULARY\n'

In [23]:
sia = SentimentIntensityAnalyzer()

In [24]:
def get_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

In [25]:
df['Sentiment'] = df['Data'].apply(get_sentiment)
df[['Labels', 'Sentiment']].head()

,Labels,Sentiment
0,alt.atheism,Negative
1,alt.atheism,Positive
2,alt.atheism,Negative
3,alt.atheism,Negative
4,alt.atheism,Positive


In [26]:
df['Sentiment'].value_counts()

,count
Sentiment,
Positive,1334
Negative,631
Neutral,35


In [27]:
sentiment_by_category = pd.crosstab(df['Labels'], df['Sentiment'])
sentiment_by_category

Sentiment,Negative,Neutral,Positive
Labels,,,
alt.atheism,42,1,57
comp.graphics,13,4,83
comp.os.ms-windows.misc,24,2,74
comp.sys.ibm.pc.hardware,21,0,79
comp.sys.mac.hardware,24,3,73
comp.windows.x,20,2,78
misc.forsale,7,8,85
rec.autos,27,1,72
rec.motorcycles,30,2,68


In [30]:
"""
THE SENTIMENT ANALYSIS SHOWS MOST OF THE DESCUSSIONS WERE POSITIVE BUT SOME BLOGS HAVING DESCUSSION ON
POLITICS AND RELEGION TEND TO HAVE NEGATIVE SENTIMENT SHOWING THE USE OF CONTROVERSIAL LANGUAGE
"""

' \nTHE SENTIMENT ANALYSIS SHOWS MOST OF THE DESCUSSIONS WERE POSITIVE BUT SOME BLOGS HAVING DESCUSSION ON\nPOLITICS AND RELEGION TEND TO HAVE NEGATIVE SENTIMENT SHOWING THE USE OF CONTROVERSIAL LANGUAGE\n'